# Домашнє завдання: Батчинг, нормалізація і стабільність навчання CNN

**Завдання 1**

- Додати `BatchNorm2d` після кожного `Conv2d`.
- Використовувати `Dropout(p=0.3)` після активації `ReLU`.
- Навчити мережу на тих самих даних протягом **10 епох**.

Для самостійного виконання використано CNN для класифікації зображень **MNIST**.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

# Фіксуємо випадковість
torch.manual_seed(42)

# Вибираємо пристрій
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# Підготовка даних MNIST

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Батчинг: модель отримує дані частинами по 64 зображення
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Кількість навчальних зображень:", len(train_dataset))
print("Кількість тестових зображень:", len(test_dataset))


## Архітектура CNN

Після **кожного `Conv2d`** використовується:

`Conv2d → BatchNorm2d → ReLU → Dropout`

Це допомагає стабілізувати навчання та зменшити перенавчання.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.features = nn.Sequential(
            # Перший згортковий блок
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.MaxPool2d(2),

            # Другий згортковий блок
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = CNN().to(device)

print(model)


In [ ]:
# Функція втрат та оптимізатор

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# За умовою завдання — 10 епох
epochs = 10

loss_history = []
accuracy_history = []


In [ ]:
# Навчання моделі

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Обнулення градієнтів
        optimizer.zero_grad()

        # Forward propagation
        outputs = model(images)

        # Обчислення помилки
        loss = criterion(outputs, labels)

        # Backward propagation
        loss.backward()

        # Оновлення ваг
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    loss_history.append(epoch_loss)
    accuracy_history.append(epoch_accuracy)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy * 100:.2f}%"
    )


In [ ]:
# Перевірка моделі на тестових даних

model.eval()

correct = 0
total = 0
test_loss = 0.0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_loss /= total
test_accuracy = correct / total

print("\n================================")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print("================================")


In [ ]:
# Графік зміни Loss

plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Зміна помилки під час навчання CNN")
plt.grid()
plt.show()


In [ ]:
# Графік точності

plt.figure(figsize=(8, 5))
plt.plot(
    range(1, epochs + 1),
    [x * 100 for x in accuracy_history],
    marker="o"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Точність CNN під час навчання")
plt.grid()
plt.show()


## Висновок

У моделі після кожного `Conv2d` додано `BatchNorm2d`, після `ReLU` використано `Dropout(p=0.3)`. Навчання виконується батчами по 64 зображення та триває 10 епох. Після навчання модель перевіряється на тестовій вибірці, а зміна `Loss` і `Accuracy` показується на графіках.